# Data Aggregation Pipeline

This notebook creates aggregated tables and business metrics for reporting and analytics.

## Execution Steps:
1. Load data from Silver layer
2. Create business-level aggregations
3. Generate summary statistics
4. Store aggregated data in Gold layer

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import json

# MS Fabric specific imports
# from fabric import FabricDataFrame
# from fabric.lakehouse import Lakehouse

print(f"Starting data aggregation at {datetime.now()}")

In [ ]:
# Load data from Silver layer
def load_from_silver(table_name):
    """
    Load data from Silver layer
    
    Args:
        table_name (str): Silver table name
    
    Returns:
        pd.DataFrame: Loaded data
    """
    silver_path = f"/lakehouse/default/Tables/silver_{table_name}"
    
    print(f"Loading data from Silver layer: {silver_path}")
    
    # In real implementation, would use:
    # df = pd.read_delta(silver_path)
    
    # Simulate loading Silver data with enriched columns
    sample_data = {
        'id': range(1, 1001),
        'customer_id': [f'CUST_{i:04d}' for i in range(1, 1001)],
        'transaction_amount': [100.0 + (i * 0.5) for i in range(1, 1001)],
        'transaction_date': [datetime.now() - timedelta(days=np.random.randint(0, 30)) for _ in range(1000)],
        'transaction_category': np.random.choice(['Small', 'Medium', 'Large', 'Very Large'], 1000),
        'transaction_year': [2024] * 1000,
        'transaction_month': np.random.choice(range(1, 13), 1000),
        'transaction_day_of_week': np.random.choice(['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'], 1000),
        'is_high_value': np.random.choice([True, False], 1000),
        'customer_segment': np.random.choice(['Premium', 'Standard', 'Basic'], 1000),
        'customer_region': np.random.choice(['North', 'South', 'East', 'West'], 1000),
        'customer_age_group': np.random.choice(['18-25', '26-35', '36-45', '46-55', '55+'], 1000)
    }
    
    return pd.DataFrame(sample_data)

# Load Silver data
silver_transactions = load_from_silver('customer_transactions')
print(f"Loaded {len(silver_transactions)} records from Silver layer")
print(silver_transactions.head())

In [ ]:
# Create daily transaction summary
def create_daily_summary(df):
    """
    Create daily transaction summary
    
    Args:
        df (pd.DataFrame): Silver transaction data
    
    Returns:
        pd.DataFrame: Daily summary
    """
    daily_summary = df.groupby(df['transaction_date'].dt.date).agg({
        'id': 'count',
        'transaction_amount': ['sum', 'mean', 'median', 'std'],
        'customer_id': 'nunique',
        'is_high_value': 'sum'
    }).round(2)
    
    # Flatten column names
    daily_summary.columns = ['_'.join(col).strip() for col in daily_summary.columns.values]
    daily_summary = daily_summary.reset_index()
    
    # Rename columns
    daily_summary.columns = [
        'transaction_date', 'total_transactions', 'total_amount', 'avg_amount',
        'median_amount', 'std_amount', 'unique_customers', 'high_value_transactions'
    ]
    
    # Add calculated metrics
    daily_summary['avg_amount_per_customer'] = (daily_summary['total_amount'] / daily_summary['unique_customers']).round(2)
    daily_summary['high_value_percentage'] = ((daily_summary['high_value_transactions'] / daily_summary['total_transactions']) * 100).round(2)
    
    print(f"Created daily summary with {len(daily_summary)} days")
    
    return daily_summary

# Create daily summary
daily_summary = create_daily_summary(silver_transactions)
print(f"\nDaily summary sample:")
print(daily_summary.head())

In [ ]:
# Create customer segment analysis
def create_customer_segment_analysis(df):
    """
    Create customer segment analysis
    
    Args:
        df (pd.DataFrame): Silver transaction data
    
    Returns:
        pd.DataFrame: Customer segment analysis
    """
    segment_analysis = df.groupby(['customer_segment', 'customer_region']).agg({
        'customer_id': 'nunique',
        'transaction_amount': ['count', 'sum', 'mean'],
        'is_high_value': 'sum'
    }).round(2)
    
    # Flatten column names
    segment_analysis.columns = ['_'.join(col).strip() for col in segment_analysis.columns.values]
    segment_analysis = segment_analysis.reset_index()
    
    # Rename columns
    segment_analysis.columns = [
        'customer_segment', 'customer_region', 'unique_customers',
        'total_transactions', 'total_amount', 'avg_transaction_amount', 'high_value_transactions'
    ]
    
    # Add calculated metrics
    segment_analysis['revenue_per_customer'] = (segment_analysis['total_amount'] / segment_analysis['unique_customers']).round(2)
    segment_analysis['transactions_per_customer'] = (segment_analysis['total_transactions'] / segment_analysis['unique_customers']).round(2)
    
    print(f"Created customer segment analysis with {len(segment_analysis)} segments")
    
    return segment_analysis

# Create segment analysis
segment_analysis = create_customer_segment_analysis(silver_transactions)
print(f"\nCustomer segment analysis sample:")
print(segment_analysis.head())

In [ ]:
# Create monthly trends
def create_monthly_trends(df):
    """
    Create monthly transaction trends
    
    Args:
        df (pd.DataFrame): Silver transaction data
    
    Returns:
        pd.DataFrame: Monthly trends
    """
    monthly_trends = df.groupby(['transaction_year', 'transaction_month']).agg({
        'id': 'count',
        'transaction_amount': ['sum', 'mean'],
        'customer_id': 'nunique',
        'is_high_value': 'sum'
    }).round(2)
    
    # Flatten column names
    monthly_trends.columns = ['_'.join(col).strip() for col in monthly_trends.columns.values]
    monthly_trends = monthly_trends.reset_index()
    
    # Rename columns
    monthly_trends.columns = [
        'year', 'month', 'total_transactions', 'total_amount',
        'avg_amount', 'unique_customers', 'high_value_transactions'
    ]
    
    # Add month name for readability
    month_names = {
        1: 'January', 2: 'February', 3: 'March', 4: 'April',
        5: 'May', 6: 'June', 7: 'July', 8: 'August',
        9: 'September', 10: 'October', 11: 'November', 12: 'December'
    }
    monthly_trends['month_name'] = monthly_trends['month'].map(month_names)
    
    # Calculate growth rates (if multiple months available)
    monthly_trends = monthly_trends.sort_values(['year', 'month'])
    monthly_trends['revenue_growth_rate'] = monthly_trends['total_amount'].pct_change() * 100
    monthly_trends['transaction_growth_rate'] = monthly_trends['total_transactions'].pct_change() * 100
    
    print(f"Created monthly trends with {len(monthly_trends)} months")
    
    return monthly_trends

# Create monthly trends
monthly_trends = create_monthly_trends(silver_transactions)
print(f"\nMonthly trends sample:")
print(monthly_trends.head())

In [ ]:
# Save aggregated data to Gold layer
def save_to_gold(df, table_name):
    """
    Save DataFrame to Gold layer in Delta format
    
    Args:
        df (pd.DataFrame): Data to save
        table_name (str): Target table name
    """
    # Add metadata
    df['_created_timestamp'] = datetime.now()
    df['_pipeline_run_id'] = f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    
    gold_path = f"/lakehouse/default/Tables/gold_{table_name}"
    
    print(f"Saving {len(df)} records to Gold layer: {gold_path}")
    
    # In real implementation, would use:
    # df.to_delta(gold_path, mode='overwrite')
    
    return gold_path

# Save all aggregated tables to Gold layer
gold_paths = []

# Save daily summary
daily_path = save_to_gold(daily_summary, 'daily_transaction_summary')
gold_paths.append(daily_path)

# Save segment analysis
segment_path = save_to_gold(segment_analysis, 'customer_segment_analysis')
gold_paths.append(segment_path)

# Save monthly trends
trends_path = save_to_gold(monthly_trends, 'monthly_transaction_trends')
gold_paths.append(trends_path)

print(f"\nAll aggregated data saved to Gold layer:")
for path in gold_paths:
    print(f"  - {path}")

print("\nData aggregation pipeline completed successfully!")
print("Gold layer tables are ready for reporting and analytics.")